# Строим LLM

## Токенизация текста

In [1]:
# Загружаем учебный текст и смотрим его размер и начало.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Tota; number of character:", len(raw_text))
print(raw_text[:99])

Tota; number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
# Делим текст на слова, пробелы и знаки пунктуации.
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
print(len(preprocessed))
print(preprocessed[:99])

9235
['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--', 'so', ' ', 'it', ' ', 'was', ' ', 'no', ' ', 'great', ' ', 'surprise', ' ', 'to', ' ', 'me', ' ', 'to', ' ', 'hear', ' ', 'that', ',', '', ' ', 'in', ' ', 'the', ' ', 'height', ' ', 'of', ' ', 'his', ' ', 'glory', ',', '', ' ', 'he', ' ', 'had', ' ', 'dropped', ' ', 'his', ' ', 'painting', ',', '', ' ', 'married', ' ', 'a', ' ', 'rich', ' ', 'widow', ',', '', ' ', 'and', ' ', 'established', ' ', 'himself', ' ', 'in', ' ', 'a']


## Преобразование токенов в идентификаторы токенов

In [3]:
# Собираем отсортированный список уникальных токенов.
all_words = sorted(set(preprocessed))
print(all_words[:99])

['', '\n', ' ', '!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed', 'Devonshire', 'Don', 'Dubarry', 'Emperors', 'Florence', 'For', 'Gallery', 'Gideon', 'Gisburn', 'Gisburns', 'Grafton', 'Greek', 'Grindle', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her', 'Hermia', 'His', 'How', 'I', 'If', 'In', 'It', 'Jack', 'Jove', 'Just', 'Lord', 'Made', 'Miss', 'Money', 'Monte', 'Moon-dancers', 'Mr', 'Mrs', 'My', 'Never', 'No', 'Now', 'Nutley', 'Of', 'Oh', 'On', 'Once', 'Only', 'Or', 'Perhaps', 'Poor', 'Professional', 'Renaissance', 'Rickham', 'Riviera', 'Rome', 'Russian', 'Sevres', 'She', 'Stroud', 'Strouds', 'Suddenly', 'That', 'The', 'Then', 'There']


In [4]:
# Строим словарь token -> id и просматриваем первые элементы.
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('', 0)
('\n', 1)
(' ', 2)
('!', 3)
('"', 4)
("'", 5)
('(', 6)
(')', 7)
(',', 8)
('--', 9)
('.', 10)
(':', 11)
(';', 12)
('?', 13)
('A', 14)
('Ah', 15)
('Among', 16)
('And', 17)
('Are', 18)
('Arrt', 19)
('As', 20)
('At', 21)
('Be', 22)
('Begin', 23)
('Burlington', 24)
('But', 25)
('By', 26)
('Carlo', 27)
('Chicago', 28)
('Claude', 29)
('Come', 30)
('Croft', 31)
('Destroyed', 32)
('Devonshire', 33)
('Don', 34)
('Dubarry', 35)
('Emperors', 36)
('Florence', 37)
('For', 38)
('Gallery', 39)
('Gideon', 40)
('Gisburn', 41)
('Gisburns', 42)
('Grafton', 43)
('Greek', 44)
('Grindle', 45)
('Grindles', 46)
('HAD', 47)
('Had', 48)
('Hang', 49)
('Has', 50)


In [5]:
# Описываем простой токенизатор на основе готового словаря.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        # Создает обратный словарь, проецирующий идентификаторы в токены
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        """Текст в идентификаторы токенов."""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Токены в текст."""
        text = " ".join([self.int_to_str[i] for i in ids])
        # Удаляет пробелы перед определенным знаком пунктуации
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
        

In [6]:
# Проверяем кодирование фразы в числовые идентификаторы.
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
    Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[4, 59, 5, 853, 991, 605, 536, 749, 8, 1129, 599, 8, 4, 70, 10, 41, 854, 1111, 757, 796, 10]


In [7]:
# Декодируем идентификаторы обратно в текст.
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [8]:
# Показываем пример слова, которого нет в словаре.
# Слово отсуствующее в словаре
# text = "Hello, do you like tea?"
# print(tokenizer.encode(text))

## Добавление контекстных токенов

In [9]:
# Добавляем специальные токены конца текста и неизвестного слова.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<[unk]>"])
print(len(vocab.items()))
vocab = {token:integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))

1133
1135


In [10]:
# Проверяем, что специальные токены попали в конец словаря.
for element in list(vocab.items())[-5:]:
    print(element)

('younger', 1130)
('your', 1131)
('yourself', 1132)
('<|endoftext|>', 1133)
('<[unk]>', 1134)


In [11]:
# Улучшаем токенизатор: добавляем обработку unknown и special tokens.
import re

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        # Более аккуратное разбиение + защита special tokens
        # Сначала заменяем special tokens на временный маркер, чтобы их не разбило
        special_tokens = ["<|endoftext|>", "<[unk]>"]
        
        for token in special_tokens:
            text = text.replace(token, f" {token} ")  # окружаем пробелами для надёжного выделения
        
        # Разбиваем
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        
        # Заменяем неизвестные токены
        ids = []
        for item in preprocessed:
            if item in self.str_to_int:
                ids.append(self.str_to_int[item])
            else:
                ids.append(self.str_to_int['<[unk]>'])
        
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        text = re.sub(r'\s+([<|])', r'\1', text)   # чистим пробелы перед special tokens
        return text

In [24]:
# Склеиваем два текста специальным маркером конца текста.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|>".join((text1, text2))
print(text)

Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.


In [25]:
# Проверяем кодирование и декодирование токенизатором V2.
tokenizer = SimpleTokenizerV2(vocab)
encoded = tokenizer.encode(text)
print(encoded)
decoded = tokenizer.decode(encoded)
print(decoded)

[1134, 8, 358, 1129, 631, 978, 13, 1133, 58, 991, 959, 987, 725, 991, 1134, 10]
<[unk]>, do you like tea?<|endoftext|> In the sunlit terraces of the<[unk]>.


## Кодирование пар байтов

In [26]:
# Подключаем готовый GPT-2 токенизатор из tiktoken.
import tiktoken 
tokenizer = tiktoken.get_encoding("gpt2")

In [27]:
# Выводим пример текста для токенизации GPT-2.
text

'Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.'

In [28]:
# Кодируем текст GPT-2 токенизатором, разрешая special token.
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
integers

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 50256,
 818,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 262,
 20562,
 13]

In [29]:
# Восстанавливаем текст из GPT-2 токенов.
strings = tokenizer.decode(integers)
strings

'Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.'

## Выборка данных с помощью контекстного окна

In [32]:
# Токенизируе весь рассказ с токенизатором BPE
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
len(enc_text)

5145

In [35]:
# Смотрим первые 50 значений токенизирванного текста
enc_sample = enc_text[:50]
enc_sample

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 1576,
 438,
 568,
 340,
 373,
 645,
 1049,
 5975,
 284,
 502,
 284,
 3285,
 326,
 11,
 287,
 262,
 6001,
 286,
 465,
 13476,
 11,
 339,
 550,
 5710,
 465,
 12036,
 11,
 6405,
 257,
 5527,
 27075,
 11]

Один из самых простых способов создания пар "входные данные - цель" для предсказания следующего слова - это создать две переменные х и у, где х содержит входные токены, а у - цели, которые являются входными токенами сдвинутыми на 1: 

In [36]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y: {y}")

x: [40, 367, 2885, 1464]
y: [367, 2885, 1464, 1807]


In [38]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Слева от стрелки - входные данные, а справа - целевые
    print(context, " ----> ", desired)

[40]  ---->  367
[40, 367]  ---->  2885
[40, 367, 2885]  ---->  1464
[40, 367, 2885, 1464]  ---->  1807


In [40]:
# Преобразование идентификаторов в текст
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Слева от стрелки - входные данные, а справа - целевые
    print(tokenizer.decode(context), " ----> ", tokenizer.decode(desired))

TypeError: argument 'tokens': 'int' object is not an instance of 'Sequence'